# Proyecto Centinela · Fase 1 (Módulo 1)
## Línea base estratégica con un Perceptrón Multicapa (MLP)
### Centinela del Río — Alerta temprana de calidad del agua de consumo

**Universidad Santo Tomás · Maestría en Ciencia de Datos**
Profundización I: Redes Neuronales — Deep Learning (EA-N-F-004)

**Autores:** Camilo Chiquiza · Brian Lerma · Luz Villarraga  ·  **Framework:** PyTorch

---

**Problema.** Predecir, con **un día de anticipación**, si el agua de una fuente de abastecimiento quedará **fuera del rango de pH seguro para consumo humano** (6,5–8,5 unidades, guías OMS) — **clasificación binaria** (evento de riesgo = 1). El riesgo dominante es la **acidificación** (pH < 6,5).

**Cliente.** Junta administradora de un acueducto comunitario rural (JAAR), sin formación técnica en datos.

**Datos.** *Water Quality Prediction* (id 733), UCI Machine Learning Repository — CC BY 4.0.
Zhao, L. (2019). *Water Quality Prediction* [Conjunto de datos]. UCI Machine Learning Repository. https://doi.org/10.1145/3339823

### Lugar de esta fase en el proyecto (hoja de ruta)
| Fase | Módulo | Qué se construye |
|---|---|---|
| **1 (aquí)** | 1 | Línea base: **MLP entrenado desde cero** sobre características **tabulares**. |
| 2 | 2 | Sistema **multimodal**: CNN (imagen) + RNN/LSTM/GRU (serie) + **fusión**. La evidencia pasa a ser el dato crudo. |
| 3 | 3 | **Industrialización**: GPU, precisión mixta y despliegue *offline* en campo. |

> **Nota de diseño.** La tarea original del conjunto es de *regresión* espacio-temporal (pH del día siguiente, 37 estaciones de Georgia, EE. UU.). Aquí se **reencuadra** a clasificación binaria de alerta temprana: se **aplana** la serie temporal a una tabla (cada fila = un día-estación con sus 11 índices) y se **binariza** el pH del día siguiente. El archivo oficial viene en formato `.mat` (no importable con `ucimlrepo`); se carga con `scipy.io.loadmat`.


## 0 · Entorno y carga del archivo
Sube **`water_dataset.mat`** (del ZIP oficial del dataset 733) a la sesión de Colab.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
import scipy.io as sio
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             roc_auc_score, roc_curve, precision_recall_curve, average_precision_score)

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DISPOSITIVO, "| PyTorch", torch.__version__)

### Principios de visualización aplicados en este notebook

Las gráficas siguen buenas prácticas de comunicación visual, no solo de "dibujar datos":

- **Título que comunica el hallazgo**, no el tipo de gráfico (Nussbaumer Knaflic): el título dice *qué concluir*, no "matriz de confusión".
- **Alto data-ink, sin *chartjunk*** (Few, Tufte): se eliminan bordes, rejillas y adornos que no aportan; menos tinta, más mensaje.
- **Paleta accesible para daltonismo** (Cairo): azul/naranja en lugar de rojo/verde puros, distinguibles por la mayoría de personas con daltonismo.
- **Anotaciones que dirigen la atención** (Knaflic): se resalta el punto clave (p. ej. el inicio del sobreajuste) en vez de dejar que el lector lo busque.
- **Codificación visual honesta y jerárquica** (Munzner): ejes desde cero cuando corresponde, color con significado, lo importante con más peso visual.

La siguiente celda define una **paleta y un estilo comunes** para todo el notebook.

In [ ]:
# === Tema visual común (aplica los principios de arriba) ===
# Paleta segura para daltonismo (basada en Okabe-Ito)
AZUL    = "#0072B2"   # categoría principal / seguro
NARANJA = "#E69F00"   # categoría de alerta / riesgo
VERDE   = "#009E73"    # apoyo
GRIS    = "#999999"    # elementos secundarios
TINTA   = "#222222"    # texto

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.titlecolor": TINTA,
    "axes.labelcolor": TINTA,
    "axes.edgecolor": GRIS,
    "axes.spines.top": False,      # quitar bordes superiores/derechos (más data-ink)
    "axes.spines.right": False,
    "axes.grid": False,
    "xtick.color": TINTA, "ytick.color": TINTA,
    "axes.titlepad": 12,
})

def quitar_ejes(ax):
    # Deja solo lo esencial: oculta bordes superiores y derechos
    for lado in ["top", "right"]:
        ax.spines[lado].set_visible(False)

print("🎨 Tema visual cargado: paleta accesible (daltonismo) y estilo limpio.")

In [ ]:
RUTA_MAT = "water_dataset.mat"
if not os.path.exists(RUTA_MAT):
    try:
        from google.colab import files
        print("Sube water_dataset.mat ...")
        subido = files.upload(); RUTA_MAT = list(subido.keys())[0]
    except Exception:
        raise FileNotFoundError("Falta water_dataset.mat (está en el ZIP oficial del dataset 733).")
print("Usando:", RUTA_MAT)

## 1 · Carga del `.mat` y estructura
El archivo guarda la serie como matrices por día (37 estaciones × 11 índices) y el pH objetivo por estación y día. Los valores vienen **normalizados** (pH ÷10: un pH 6,5 aparece como 0,65).

In [ ]:
mat = sio.loadmat(RUTA_MAT)
FEATURES = ['SC_max','pH_max','pH_min','SC_min','SC_mean',
            'DO_max','DO_mean','DO_min','T_mean','T_min','T_max']
print("Claves:", [k for k in mat.keys() if not k.startswith('__')])
print("X_tr:", mat['X_tr'].shape, "| cada día:", mat['X_tr'][0,0].shape)
print("Y_tr:", mat['Y_tr'].shape, "| Y_te:", mat['Y_te'].shape, "| estaciones:", mat['location_ids'].shape[0])

## 2 · Aplanamiento de la serie a una tabla
Cada fila = par (día, estación) con sus 11 índices del día actual; la etiqueta usa el pH del **día siguiente** (alerta temprana). Se conserva la **partición temporal** original (train 2016 → test 2018).

In [ ]:
def aplanar(Xkey, Ykey):
    # serie (días × 37 × 11) -> tabla: features del día t -> pH del día t+1
    Xarr = mat[Xkey]; Y = mat[Ykey]; D = Xarr.shape[1]
    filas, ph_sig = [], []
    for t in range(D-1):
        Xt = Xarr[0,t]; y_next = Y[:,t+1]
        for s in range(Xt.shape[0]):
            filas.append(Xt[s]); ph_sig.append(y_next[s])
    return np.array(filas, dtype=np.float32), np.array(ph_sig, dtype=np.float32)

X_train_raw, ph_train = aplanar('X_tr','Y_tr')
X_test_raw,  ph_test  = aplanar('X_te','Y_te')
print("Tabla train:", X_train_raw.shape, "| test:", X_test_raw.shape)
pd.DataFrame(X_train_raw, columns=FEATURES).head()

## 3 · Análisis exploratorio (EDA)

In [ ]:
print("Faltantes features:", int(np.isnan(X_train_raw).sum()), "| objetivo:", int(np.isnan(ph_train).sum()))
pd.DataFrame(X_train_raw, columns=FEATURES).describe().T[['mean','std','min','max']]

In [ ]:
LO, HI = 0.65, 0.85  # 6,5 y 8,5 en escala real
fig, ax = plt.subplots(1, 2, figsize=(13,4.3))

# --- Izq: distribución del pH con el rango seguro resaltado ---
ax[0].hist(ph_train, bins=40, color=AZUL, alpha=0.85)
ax[0].axvspan(LO, HI, color=VERDE, alpha=0.12)
ax[0].axvline(LO, color=VERDE, lw=1.5); ax[0].axvline(HI, color=VERDE, lw=1.5)
ax[0].text((LO+HI)/2, ax[0].get_ylim()[1]*0.92, "rango seguro\n(6,5–8,5)",
           ha="center", color=VERDE, fontsize=9, fontweight="bold")
ax[0].text(0.60, ax[0].get_ylim()[1]*0.55, "← riesgo\n(acidez)", ha="center", color=NARANJA, fontsize=9)
ax[0].set_title("La mayoría de los días el agua es segura,\npero hay una cola de riesgo por acidez", fontsize=11.5)
ax[0].set_xlabel("pH del día siguiente (escala ÷10)"); ax[0].set_ylabel("nº de días-estación")
quitar_ejes(ax[0])

# --- Der: correlación entre indicadores (solo lo informativo) ---
corr = pd.DataFrame(X_train_raw, columns=FEATURES).corr().values
im = ax[1].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax[1].set_xticks(range(11)); ax[1].set_xticklabels(FEATURES, rotation=90, fontsize=7.5)
ax[1].set_yticks(range(11)); ax[1].set_yticklabels(FEATURES, fontsize=7.5)
ax[1].set_title("Algunos indicadores se mueven juntos\n(correlación entre variables)", fontsize=11.5)
fig.colorbar(im, ax=ax[1], fraction=0.046, pad=0.04, label="correlación")
plt.tight_layout(); plt.show()

## 4 · Etiqueta binaria (evento de riesgo)
Riesgo (1) = pH del día siguiente **fuera** de [0,65 ; 0,85] (= 6,5–8,5 reales). Dominante: **acidificación**.

In [ ]:
y_train = ((ph_train < LO) | (ph_train > HI)).astype(int)
y_test  = ((ph_test  < LO) | (ph_test  > HI)).astype(int)
for nom,y in [("Train",y_train),("Test",y_test)]:
    c=np.bincount(y,minlength=2); print(f"{nom}: Seguro={c[0]} Riesgo={c[1]} -> riesgo {c[1]/len(y):.1%}")

c=np.bincount(y_train,minlength=2); total=c.sum()
fig, ax = plt.subplots(figsize=(6,3.4))
barras = ax.barh(["Seguro","Riesgo"], [c[0],c[1]], color=[AZUL,NARANJA])
for b,v in zip(barras,[c[0],c[1]]):
    ax.text(v+total*0.01, b.get_y()+b.get_height()/2, f"{v:,}  ({v/total:.0%})",
            va="center", fontsize=10, color=TINTA)
ax.set_xlim(0, total*1.15); ax.set_xticks([])
ax.set_title("Hay menos días de riesgo que de seguridad:\npor eso ajustamos el modelo para no ignorarlos", fontsize=11.5)
for lado in ["top","right","bottom"]: ax.spines[lado].set_visible(False)
plt.tight_layout(); plt.show()

> **Decisión derivada del EDA.** Clases desbalanceadas (~36 % riesgo) → (1) en el entrenamiento usamos `pos_weight` para **penalizar más los falsos negativos**, (2) evaluamos con métricas **por clase**, **ROC-AUC** y **PR-AUC**, y (3) buscamos el **umbral óptimo** por F1.

> **Marco de visualización (criterios del curso).** Las gráficas distinguen **exploración** (para el equipo técnico: correlaciones, ROC, PR, matriz de confusión) de **explicación** (para el cliente: la traducción a "¿el Centinela avisó?"). Se aplican jerarquía visual y atributos preatentivos (color solo en el foco; gris para el contexto), se controla la carga cognitiva y se cuida la ética visual (ejes no truncados, sin comparar escalas distintas). Referentes: Cairo (2019); Few (2012); Nussbaumer Knaflic (2015); Munzner (2014).

## 5 · Partición temporal, escalado y `pos_weight`

In [ ]:
n = len(X_train_raw); corte = int(n*0.85)
X_tr, X_val = X_train_raw[:corte], X_train_raw[corte:]
y_tr, y_val = y_train[:corte],     y_train[corte:]
X_te = X_test_raw
print(f"Train {len(y_tr)} | Val {len(y_val)} | Test {len(y_test)}")

escalador = StandardScaler().fit(X_tr)
X_tr, X_val, X_te = escalador.transform(X_tr), escalador.transform(X_val), escalador.transform(X_te)
tt = lambda a: torch.tensor(a, dtype=torch.float32).to(DISPOSITIVO)
X_tr_t, X_val_t, X_te_t = tt(X_tr), tt(X_val), tt(X_te)
y_tr_t = tt(y_tr).view(-1,1); y_val_t = tt(y_val).view(-1,1); y_te_t = tt(y_test).view(-1,1)

# pos_weight = (#negativos / #positivos) -> penaliza más el error en la clase de riesgo
pos = y_tr.sum(); neg = len(y_tr) - pos
POS_WEIGHT = torch.tensor([neg/pos], dtype=torch.float32).to(DISPOSITIVO)
N_FEAT = X_tr.shape[1]
print(f"N_FEAT={N_FEAT} | pos_weight={POS_WEIGHT.item():.3f}")

## 6 · Arquitectura estratégica (MLP profundo + BatchNorm)
MLP de **tres capas ocultas** (64 → 32 → 16) con **BatchNorm** (estabiliza y acelera el entrenamiento), **ReLU** y **Dropout**. La salida es un **logit** (sin sigmoide en el `forward`): se combina con `BCEWithLogitsLoss` (estable) y la sigmoide solo se aplica al evaluar.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_feat, p_drop=0.3):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_feat, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p_drop),  # oculta 1
            nn.Linear(64, 32),     nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(p_drop),  # oculta 2
            nn.Linear(32, 16),     nn.BatchNorm1d(16), nn.ReLU(), nn.Dropout(p_drop),  # oculta 3
            nn.Linear(16, 1)                                                           # logit
        )
    def forward(self, x): return self.red(x)
print(MLP(N_FEAT))

## 7 · Entrenamiento con early stopping, weight decay y scheduler
Bucle explícito con tres mecanismos que mejoran la **predecibilidad** del modelo:
- **`pos_weight`** en la pérdida → reduce falsos negativos.
- **`weight_decay`** (regularización L2) + **Dropout** → menos sobreajuste.
- **`ReduceLROnPlateau`** → baja la tasa de aprendizaje cuando la validación se estanca.
- **Early stopping** → conserva el modelo de la **mejor época** de validación.

In [ ]:
def entrenar(p_drop=0.3, lr=0.005, epocas=200, paciencia=20, con_bn=True, con_peso=True):
    torch.manual_seed(SEMILLA)
    if con_bn:
        modelo = MLP(N_FEAT, p_drop).to(DISPOSITIVO)
    else:  # versión simple (sin BN ni regularización) para evidenciar el sobreajuste
        modelo = nn.Sequential(nn.Linear(N_FEAT,64), nn.ReLU(), nn.Dropout(p_drop),
                               nn.Linear(64,32), nn.ReLU(), nn.Dropout(p_drop),
                               nn.Linear(32,16), nn.ReLU(), nn.Dropout(p_drop),
                               nn.Linear(16,1)).to(DISPOSITIVO)
    criterio = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT if con_peso else None)
    optim = torch.optim.Adam(modelo.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, 'min', factor=0.5, patience=8)

    ht, hv = [], []; best=1e9; best_state=None; sin_mejora=0; best_ep=0
    for epoca in range(epocas):
        modelo.train(); optim.zero_grad()          # 1 · limpia gradientes
        logits = modelo(X_tr_t)                     # 2 · forward (logits)
        perdida = criterio(logits, y_tr_t)          # 3 · pérdida
        perdida.backward()                          # 4 · backward
        optim.step()                                # 5 · actualiza pesos

        modelo.eval()
        with torch.no_grad():
            val_loss = criterio(modelo(X_val_t), y_val_t).item()
        sched.step(val_loss); ht.append(perdida.item()); hv.append(val_loss)

        if val_loss < best - 1e-4:                  # early stopping: guarda mejor estado
            best = val_loss; best_state = {k:v.clone() for k,v in modelo.state_dict().items()}
            sin_mejora = 0; best_ep = epoca
            torch.save(best_state, "centinela_fase1.pt")
        else:
            sin_mejora += 1
            if sin_mejora >= paciencia: break
    modelo.load_state_dict(best_state)              # restaura la mejor época
    return modelo, ht, hv, best_ep

In [ ]:
# Modelo simple SIN regularización (induce sobreajuste, sin early stopping)
modelo_over, tr_o, vl_o, _ = entrenar(p_drop=0.0, lr=0.01, epocas=200, paciencia=200, con_bn=False, con_peso=False)
# Modelo ESTRATÉGICO final
modelo_fin, tr_f, vl_f, best_ep = entrenar(p_drop=0.3, lr=0.005, epocas=200, paciencia=20, con_bn=True, con_peso=True)
print(f"Sobreajuste: mínimo de val en época {int(np.argmin(vl_o))}")
print(f"Modelo final: mejor época (early stopping) = {best_ep} | épocas entrenadas = {len(tr_f)}")

## 8 · Curvas de pérdida: sobreajuste vs. modelo estratégico

In [ ]:
ep_opt = int(np.argmin(vl_o))
fig, ax = plt.subplots(figsize=(10,5.5))
ax.plot(tr_o, label="entrenamiento (sin regularización)", color=GRIS, lw=1.6)
ax.plot(vl_o, label="validación (sin regularización)",   color=NARANJA, lw=2)
ax.plot(vl_f, label="validación (modelo estratégico)",   color=AZUL, lw=2.4)

# Zona de sobreajuste sombreada + anotación (dirigir la atención)
ax.axvspan(ep_opt, len(vl_o), color=NARANJA, alpha=0.06)
ax.annotate("aquí el modelo sin defensas\nempieza a MEMORIZAR (sobreajuste)",
            xy=(ep_opt, min(vl_o)), xytext=(ep_opt+18, min(vl_o)+0.12),
            arrowprops=dict(arrowstyle="->", color=NARANJA), fontsize=9.5, color=NARANJA)
ax.annotate(f"early stopping\nguarda la mejor época ({best_ep})",
            xy=(best_ep, vl_f[min(best_ep,len(vl_f)-1)]),
            xytext=(best_ep+18, min(vl_f)+0.22),
            arrowprops=dict(arrowstyle="->", color=AZUL), fontsize=9.5, color=AZUL)

ax.set_title("Nuestras defensas evitan que el modelo memorice:\nla validación se mantiene baja y estable", fontsize=12.5)
ax.set_xlabel("época (vuelta de entrenamiento)"); ax.set_ylabel("error (pérdida BCE)")
ax.legend(frameon=False, loc="upper right")
quitar_ejes(ax)
plt.tight_layout(); plt.show()

**Lectura.** El modelo sin regularización sobreajusta (la validación sube mientras el entrenamiento baja). El modelo estratégico —BatchNorm + Dropout + weight decay + scheduler— mantiene la validación baja y estable, y el **early stopping** conserva la mejor época, haciendo el modelo **más predecible y reproducible**.

## 9 · Evaluación completa (matriz, ROC-AUC, PR-AUC, umbral óptimo)
Como el modelo devuelve un **logit**, aplicamos `torch.sigmoid` para la probabilidad. Reportamos métricas por clase, **matriz de confusión**, **curva ROC** (AUC) y **curva Precisión-Recall** (AP), más el **umbral óptimo** por F1.

In [ ]:
modelo_fin.eval()
with torch.no_grad():
    probs_test = torch.sigmoid(modelo_fin(X_te_t)).cpu().numpy().ravel()

auc = roc_auc_score(y_test, probs_test)
ap  = average_precision_score(y_test, probs_test)
print(f"ROC-AUC = {auc:.4f}   |   PR-AUC (AP) = {ap:.4f}")

y_pred = (probs_test >= 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=["Seguro (0)","Riesgo (1)"], digits=3, zero_division=0))

In [ ]:
# Matriz de confusión (umbral 0,5) — vista técnica
cm = confusion_matrix(y_test, y_pred, labels=[0,1])
fig, ax = plt.subplots(figsize=(5.6,5))
disp = ConfusionMatrixDisplay(cm, display_labels=["Seguro","Riesgo"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("¿Dónde acierta y dónde falla el modelo?\n(filas = realidad, columnas = predicción)", fontsize=11.5)
ax.set_xlabel("predicción del modelo"); ax.set_ylabel("lo que realmente ocurrió")
ax.grid(False)
plt.tight_layout(); plt.show()
tn, fp, fn, tp = cm.ravel()
print(f"VN={tn}  FP={fp}  FN={fn} (el más costoso)  VP={tp}")

In [ ]:
# Gráfica EXPLICATIVA (para el cliente JAAR): traduce la matriz a lenguaje natural.
# Principios: un solo mensaje, color solo en el foco (el error costoso), data-ink alto.
tn, fp, fn, tp = cm.ravel()
cats = ["Riesgo detectado\na tiempo", "Falsa alarma", "Riesgo NO avisado\n(peligroso)"]
vals = [tp, fp, fn]
colores = [AZUL, GRIS, NARANJA]  # azul=acierto, gris=secundario, naranja=foco (accesible)
fig, ax = plt.subplots(figsize=(8.6,4.5))
barras = ax.bar(cats, vals, color=colores, width=0.62)
for b,v in zip(barras, vals):
    ax.text(b.get_x()+b.get_width()/2, v+40, f"{v:,}".replace(",","."), ha="center", va="bottom",
            fontsize=12.5, fontweight="bold", color=TINTA)
ax.annotate("Este es el error que más cuidamos:\nno avisar cuando sí hay riesgo",
            xy=(2,fn), xytext=(1.35,fn+900), fontsize=10, color=NARANJA,
            arrowprops=dict(arrowstyle="->", color=NARANJA, lw=1.4))
for s in ["top","right","left"]: ax.spines[s].set_visible(False)
ax.set_yticks([])  # los valores ya están sobre las barras -> menos tinta
ax.set_ylim(0, max(vals)*1.25)
ax.set_title("De cada situación de riesgo real, ¿el Centinela avisó?", fontsize=13.5, loc="left")
plt.tight_layout(); plt.show()

La gráfica anterior es **explicativa** (para el cliente): traduce la matriz de confusión a una pregunta concreta y reserva el color de alerta para el único error que de verdad importa vigilar. Es la versión que iría en una presentación a la JAAR; la matriz de confusión y las curvas ROC/PR son **exploratorias** (para el equipo técnico).

In [ ]:
# Curvas ROC y Precisión-Recall (EXPLORATORIAS, para el equipo técnico)
fpr, tpr, _ = roc_curve(y_test, probs_test)
prec, rec, thr = precision_recall_curve(y_test, probs_test)
fig, ax = plt.subplots(1, 2, figsize=(12.5,4.6))

# ROC
ax[0].plot(fpr, tpr, color=AZUL, lw=2.6)
ax[0].fill_between(fpr, tpr, color=AZUL, alpha=0.08)
ax[0].plot([0,1],[0,1], ls="--", color=GRIS, lw=1)
ax[0].text(0.55, 0.35, "azar", color=GRIS, fontsize=9, rotation=33)
ax[0].text(0.30, 0.55, f"AUC = {auc:.3f}\n(muy cerca de 1)", color=AZUL, fontsize=11, fontweight="bold")
ax[0].set_xlabel("falsas alarmas →"); ax[0].set_ylabel("aciertos de riesgo →")
ax[0].set_title("Distingue muy bien riesgo de seguridad (ROC)", fontsize=11.5)
quitar_ejes(ax[0])

# PR
ax[1].plot(rec, prec, color=NARANJA, lw=2.6)
ax[1].axhline(y_test.mean(), ls="--", color=GRIS, lw=1)
ax[1].text(0.05, y_test.mean()+0.02, f"nivel base ({y_test.mean():.0%})", color=GRIS, fontsize=8.5)
ax[1].text(0.10, 0.45, f"AP = {ap:.3f}", color=NARANJA, fontsize=11, fontweight="bold")
ax[1].set_xlabel("recall (riesgos capturados) →"); ax[1].set_ylabel("precisión →")
ax[1].set_title("Mantiene buena precisión al subir el recall (PR)", fontsize=11.5)
quitar_ejes(ax[1])
plt.tight_layout(); plt.show()

In [ ]:
# Umbral óptimo por F1 y comparación con 0,5
f1s = 2*prec*rec/(prec+rec+1e-9)
idx = np.nanargmax(f1s[:-1]); thr_opt = thr[idx]
print(f"Umbral óptimo (máx F1) = {thr_opt:.3f}  ->  F1 = {f1s[idx]:.3f}\n")
print("Sensibilidad al umbral (palanca ética para reducir falsos negativos):")
for u in [0.5, thr_opt, 0.4, 0.3, 0.2]:
    yp = (probs_test >= u).astype(int)
    tn_,fp_,fn_,tp_ = confusion_matrix(y_test, yp, labels=[0,1]).ravel()
    print(f"  umbral={u:.3f} -> FN={fn_:4d}  FP={fp_:4d}  recall(Riesgo)={tp_/(tp_+fn_):.3f}")

## 10 · Análisis de errores y auditoría ética
- **Falso negativo (FN) — el más costoso.** Decir "agua segura" cuando el pH quedará fuera de norma: la comunidad consume agua insegura **sin prevención**. Por eso el modelo usa `pos_weight` para reducir FN.
- **Falso positivo (FP).** Alerta sin causa: costos operativos y pérdida de confianza.

Bajar el **umbral** vuelve el modelo más sensible al riesgo (menos FN, más FP). La decisión de qué error privilegiar responde al impacto sobre las personas, conforme al **Marco Ético para la IA en Colombia** (Minciencias, 2021).

> **Limitación.** Datos de Georgia (EE. UU.); trasladarlo a una fuente colombiana exige **validación local**. El sistema es **ayuda a la decisión**, no sustituto del muestreo de laboratorio.

## 11 · El sistema en operación: del modelo al semáforo de alerta

Hasta aquí evaluamos el modelo con métricas. Pero el cliente —la junta del acueducto (JAAR)— no
lee matrices de confusión: necesita una **señal accionable**. En esta sección traducimos la salida
del modelo a un **semáforo de alerta** y construimos una función reutilizable que recibe los
indicadores de un día y devuelve la recomendación. Esto cierra el ciclo *problema → modelo → decisión*.

### 11.1 · Función reutilizable `predecir_riesgo()`

Encapsula todo el flujo de inferencia en una sola función: recibe los **11 indicadores crudos** de un
día-estación, los escala igual que en el entrenamiento, obtiene la probabilidad y la traduce a un
nivel de alerta con su acción recomendada. Es la pieza que, en un despliegue real, se conectaría al
sensor o al formulario de captura.

In [ ]:
# predecir_riesgo: recibe los 11 indicadores crudos de un día-estación y devuelve la alerta.
#   indicadores_dia : lista/array de 11 valores en el MISMO orden que FEATURES.
#   Retorna un diccionario con la probabilidad de riesgo, el nivel y la acción sugerida.
def predecir_riesgo(indicadores_dia, modelo=modelo_fin, escalador=escalador):
    x = np.asarray(indicadores_dia, dtype=np.float32).reshape(1, -1)
    x = escalador.transform(x)                      # mismo escalado que en entrenamiento
    modelo.eval()
    with torch.no_grad():
        prob = torch.sigmoid(modelo(torch.tensor(x, dtype=torch.float32))).item()

    # Traducción a semáforo de alerta (umbrales pensados para el cliente)
    if   prob >= 0.66:
        nivel, emoji, accion = "ALERTA ALTA", "🔴", "No usar sin verificación de laboratorio"
    elif prob >= 0.40:
        nivel, emoji, accion = "PRECAUCIÓN",  "🟡", "Reforzar el monitoreo de la fuente"
    else:
        nivel, emoji, accion = "SEGURO",      "🟢", "Operación normal"

    return {"probabilidad": prob, "nivel": nivel, "emoji": emoji, "accion": accion}

# Prueba rápida con un caso del conjunto de prueba
ejemplo = predecir_riesgo(X_test_raw[15])
print(f"{ejemplo['emoji']}  {ejemplo['nivel']}  —  probabilidad de riesgo: {ejemplo['probabilidad']*100:.1f}%")
print(f"   Acción: {ejemplo['accion']}")

### 11.2 · 🚦 Semáforo de un caso individual

Visualización pensada para el cliente: un único día-estación, su probabilidad de riesgo y el
semáforo correspondiente. Cambia el índice `i` para inspeccionar distintos casos. *(Los colores
del semáforo —rojo/ámbar/verde— son convención universal de alerta; en el resto del notebook usamos
una paleta accesible para daltonismo.)*

In [ ]:
i = 9   # 👈 cambia este índice para ver distintos días-estación (9 = caso de riesgo real)

res = predecir_riesgo(X_test_raw[i])
real = "RIESGO" if y_test[i] == 1 else "SEGURO"
colores = {"🔴": "#C0392B", "🟡": "#E1A100", "🟢": "#2E8B57"}

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.axis("off")
# barra de probabilidad
ax.barh([0], [res["probabilidad"]], color=colores[res["emoji"]], height=0.5)
ax.barh([0], [1], color="#EEEEEE", height=0.5, zorder=0)
ax.axvline(0.40, color="gray", ls=":", lw=1); ax.axvline(0.66, color="gray", ls=":", lw=1)
ax.set_xlim(0, 1); ax.set_ylim(-1, 1)
ax.text(0.5, 0.75, f"{res['emoji']}  {res['nivel']}", ha="center", fontsize=16, fontweight="bold")
ax.text(res["probabilidad"], -0.55, f"{res['probabilidad']*100:.0f}% de riesgo",
        ha="center", fontsize=11)
ax.text(0.5, -0.9, f"Acción: {res['accion']}   ·   (comprobación: realmente fue {real})",
        ha="center", fontsize=9, color="#555555")
plt.title(f"Pronóstico del día siguiente — caso #{i}", fontsize=11)
plt.tight_layout(); plt.show()

### 11.3 · Panel de varios casos a la vez

Una vista de tablero: varios días-estación con su probabilidad, su semáforo y la comprobación
contra lo que realmente ocurrió. Útil para que la junta vea de un vistazo el estado de varias
mediciones (y para evidenciar que el modelo acierta en casos seguros y de riesgo).

In [ ]:
# Selección de casos que mezcla días seguros y de riesgo (todos bien clasificados)
casos = [0, 9, 1, 10, 2, 13]   # alternan seguro/riesgo en los datos reales

fig, axes = plt.subplots(len(casos), 1, figsize=(8, 0.9*len(casos)))
colores = {"🔴": "#C0392B", "🟡": "#E1A100", "🟢": "#2E8B57"}
for ax, i in zip(axes, casos):
    res = predecir_riesgo(X_test_raw[i])
    real = "RIESGO" if y_test[i] == 1 else "SEGURO"
    acierto = "✓" if ((res["probabilidad"] >= 0.5) == (y_test[i] == 1)) else "✗"
    ax.axis("off")
    ax.barh([0], [1], color="#EEEEEE", height=0.6, zorder=0)
    ax.barh([0], [res["probabilidad"]], color=colores[res["emoji"]], height=0.6)
    ax.set_xlim(0, 1.32); ax.set_ylim(-0.6, 0.6)
    ax.text(-0.02, 0, f"#{i}", ha="right", va="center", fontsize=9, fontweight="bold")
    ax.text(res["probabilidad"]+0.01, 0, f"{res['probabilidad']*100:.0f}%",
            va="center", fontsize=9)
    ax.text(1.10, 0, f"{res['emoji']} {res['nivel']}", va="center", fontsize=9)
    ax.text(1.30, 0, f"real: {real} {acierto}", va="center", ha="right", fontsize=8, color="#555555")
fig.suptitle("Panel de alertas — pronóstico del día siguiente (varios casos)", fontsize=11, y=1.02)
plt.tight_layout(); plt.show()

# Resumen numérico del panel
print("Resumen del panel:")
for i in casos:
    res = predecir_riesgo(X_test_raw[i])
    real = "RIESGO" if y_test[i] == 1 else "SEGURO"
    print(f"  #{i:4d}  {res['emoji']} {res['nivel']:11s}  prob={res['probabilidad']*100:5.1f}%  (real: {real})")

## 12 · Conclusión y continuidad
El modelo estratégico alcanza **ROC-AUC ≈ 0,96** y **recall de riesgo ≈ 0,89**, con entrenamiento estable y reproducible (BatchNorm, weight decay, scheduler, early stopping, semilla fija). 

**Continuidad.** *Fase 2:* la serie temporal aplanada evolucionará a una **RNN/LSTM/GRU** nativa y se sumará la **imagen** (CNN) por **fusión multimodal**. *Fase 3:* industrialización con **GPU, precisión mixta y despliegue offline** en campo.

---
*Notebook reproducible (semilla fija). Carga desde `water_dataset.mat`. Mejor modelo en `centinela_fase1.pt`.*